In [1]:
%pip install streamlit pandas plotly requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit as st
import pandas as pd
import plotly
import requests
import dotenv

print("Streamlit:", st.__version__)
print("Pandas:", pd.__version__)
print("Plotly:", plotly.__version__)
print("Requests:", requests.__version__)
print("Setup successful.")

Streamlit: 1.51.0
Pandas: 2.2.2
Plotly: 5.22.0
Requests: 2.32.2
Setup successful.


In [4]:
from pathlib import Path

secrets_file = Path(".streamlit/secrets.toml")
gitignore_file = Path(".gitignore")

print("Secrets file exists:", secrets_file.exists())
print("Gitignore exists:", gitignore_file.exists())


Secrets file exists: False
Gitignore exists: True


In [5]:
from pathlib import Path
from getpass import getpass
import json

streamlit_folder = Path(".streamlit")
streamlit_folder.mkdir(exist_ok=True)

api_key = getpass("Paste your OpenWeather API key: ")

secrets_file = streamlit_folder / "secrets.toml"
secrets_file.write_text(
    "OPENWEATHER_API_KEY = " + json.dumps(api_key) + "\n",
    encoding="utf-8"
)

del api_key

print("Secrets file exists:", secrets_file.exists())
print("Saved as:", secrets_file)

Paste your OpenWeather API key:  ········


Secrets file exists: True
Saved as: .streamlit\secrets.toml


In [1]:
from pathlib import Path
import streamlit as st

gitignore_file = Path(".gitignore")
required_entries = [
    ".streamlit/secrets.toml",
    ".env",
    ".ipynb_checkpoints/",
    "__pycache__/"
]

if not gitignore_file.exists():
    gitignore_file.write_text(
        "\n".join(required_entries) + "\n",
        encoding="utf-8"
    )

gitignore_text = gitignore_file.read_text(encoding="utf-8")

print(
    "API key loaded safely:",
    bool(st.secrets.get("OPENWEATHER_API_KEY"))
)
print(
    "Secrets file protected:",
    ".streamlit/secrets.toml" in gitignore_text
)

PermissionError: [Errno 13] Permission denied: '.gitignore'

In [3]:
from pathlib import Path

gitignore_path = Path(".gitignore")

if gitignore_path.is_dir():
    contents = list(gitignore_path.iterdir())

    if contents:
        raise RuntimeError(
            "The .gitignore folder is not empty. Stop and inspect it before continuing."
        )

    gitignore_path.rmdir()
    print("Removed the empty .gitignore folder.")

required_entries = [
    ".streamlit/secrets.toml",
    ".env",
    ".ipynb_checkpoints/",
    "__pycache__/"
]

gitignore_path.write_text(
    "\n".join(required_entries) + "\n",
    encoding="utf-8"
)

print("Gitignore is now a file:", gitignore_path.is_file())

Removed the empty .gitignore folder.
Gitignore is now a file: True


In [4]:
import streamlit as st

gitignore_text = Path(".gitignore").read_text(encoding="utf-8")

print(
    "API key loaded safely:",
    bool(st.secrets.get("OPENWEATHER_API_KEY"))
)
print(
    "Secrets file protected:",
    ".streamlit/secrets.toml" in gitignore_text
)

API key loaded safely: True
Secrets file protected: True


In [6]:
import requests
import streamlit as st

api_key = st.secrets["OPENWEATHER_API_KEY"]
city_query = "Shreveport,LA,US"

# Convert the location into coordinates
geocoding_response = requests.get(
    "https://api.openweathermap.org/geo/1.0/direct",
    params={
        "q": city_query,
        "limit": 1,
        "appid": api_key
    },
    timeout=10
)

if geocoding_response.status_code != 200:
    print(
        "Geocoding request failed with status:",
        geocoding_response.status_code
    )
else:
    locations = geocoding_response.json()

    if not locations:
        print("Location not found.")
    else:
        location = locations[0]
        latitude = location["lat"]
        longitude = location["lon"]

        weather_response = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={
                "lat": latitude,
                "lon": longitude,
                "appid": api_key,
                "units": "imperial"
            },
            timeout=10
        )

        if weather_response.status_code != 200:
            print(
                "Weather request failed with status:",
                weather_response.status_code
            )
        else:
            weather = weather_response.json()

            print("Location:", weather["name"])
            print("Temperature:", weather["main"]["temp"], "°F")
            print("Feels like:", weather["main"]["feels_like"], "°F")
            print("Humidity:", weather["main"]["humidity"], "%")
            print("Wind speed:", weather["wind"]["speed"], "mph")
            print("Conditions:", weather["weather"][0]["description"])
            print("API connection successful.")

Location: Bossier City
Temperature: 97.99 °F
Feels like: 110.59 °F
Humidity: 55 %
Wind speed: 6.55 mph
Conditions: broken clouds
API connection successful.


In [7]:
%%writefile app.py

import requests
import streamlit as st


st.set_page_config(
    page_title="Weather Risk Dashboard",
    page_icon="🌦️",
    layout="wide",
)

st.title("Weather Risk & Operations Dashboard")
st.write("Search for a location to retrieve current weather conditions.")

api_key = st.secrets["OPENWEATHER_API_KEY"]


def get_coordinates(location_query):
    response = requests.get(
        "https://api.openweathermap.org/geo/1.0/direct",
        params={
            "q": location_query,
            "limit": 1,
            "appid": api_key,
        },
        timeout=10,
    )

    if response.status_code != 200:
        return None, "The location service is currently unavailable."

    locations = response.json()

    if not locations:
        return None, "Location not found. Try adding a state or country."

    return locations[0], None


def get_current_weather(latitude, longitude, units):
    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": latitude,
            "lon": longitude,
            "appid": api_key,
            "units": units,
        },
        timeout=10,
    )

    if response.status_code != 200:
        return None, "Current weather data could not be retrieved."

    return response.json(), None


with st.form("weather_search"):
    location_query = st.text_input(
        "Location",
        value="Shreveport, LA, US",
        help="Enter a city, state, and country when possible.",
    )

    unit_choice = st.selectbox(
        "Temperature units",
        options=["Fahrenheit", "Celsius"],
    )

    submitted = st.form_submit_button("Analyze Weather")


if submitted:
    if not location_query.strip():
        st.warning("Please enter a location.")
    else:
        units = "imperial" if unit_choice == "Fahrenheit" else "metric"
        temperature_symbol = "°F" if units == "imperial" else "°C"
        wind_unit = "mph" if units == "imperial" else "m/s"

        location, location_error = get_coordinates(location_query)

        if location_error:
            st.error(location_error)
        else:
            weather, weather_error = get_current_weather(
                location["lat"],
                location["lon"],
                units,
            )

            if weather_error:
                st.error(weather_error)
            else:
                st.subheader(
                    f"{weather['name']}, {weather['sys']['country']}"
                )

                condition = weather["weather"][0]["description"].title()
                st.write(f"**Current conditions:** {condition}")

                col1, col2, col3, col4 = st.columns(4)

                col1.metric(
                    "Temperature",
                    f"{weather['main']['temp']:.1f}{temperature_symbol}",
                )
                col2.metric(
                    "Feels Like",
                    f"{weather['main']['feels_like']:.1f}{temperature_symbol}",
                )
                col3.metric(
                    "Humidity",
                    f"{weather['main']['humidity']}%",
                )
                col4.metric(
                    "Wind Speed",
                    f"{weather['wind']['speed']:.1f} {wind_unit}",
                )

                col5, col6 = st.columns(2)

                col5.metric(
                    "Visibility",
                    f"{weather.get('visibility', 0) / 1000:.1f} km",
                )
                col6.metric(
                    "Atmospheric Pressure",
                    f"{weather['main']['pressure']} hPa",
                )

                st.success("Live weather data retrieved successfully.")
                st.caption("Weather data provided by OpenWeather.")

Writing app.py


In [8]:
from pathlib import Path

print(Path.cwd())

C:\Users\PawPaw\OneDrive\Desktop\Data Science Projects\Weather Project Live API


In [9]:
%%writefile requirements.txt
streamlit==1.51.0
pandas==2.2.2
plotly==5.22.0
requests==2.32.2

Writing requirements.txt
